### rag pipeline - data ingestion to vectorDB pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

d:\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
###read all the pdfs inisde the directory
def process_all_pdfs(pdf_directory):
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    #find all pdfs recursively
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} pdf files to process")
    for pdf_file in pdf_files:
        print(f"\nProcessing : {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file)) 
            documents=loader.load()

            ##add source info to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
            
            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")

        except Exception as e:
            print(f" error : {e}")
        
    print(f"\n total doc loaded:{len(all_documents)}")
    return all_documents
all_pdf_documents=process_all_pdfs("../data")

Found 3 pdf files to process

Processing : Indrasena_2548561_ETE-DL QP.pdf
loaded 16 pages

Processing : Indrasena_561_NLP_ETE.pdf
loaded 11 pages

Processing : Satwik resume.pdf
loaded 2 pages

 total doc loaded:29


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Indrasena_2548561_ETE-DL QP', 'source': '..\\data\\pdfs\\Indrasena_2548561_ETE-DL QP.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Indrasena_2548561_ETE-DL QP.pdf', 'file_type': 'pdf'}, page_content='CHRIST  (Deemed  to  be  University),  Bangalore  –  560  029   Department  of  Computer  Science   END-TRIMESTER  EXAMINATION  MARCH-  2026   PG  III  Trimester   \nProgramme  Name:  MSAIM  Max.  Marks:  100  Course  Name:  -  Deep  Learning  Time:  3  Hrs  \nCourse\n \nCode:\n \nMAI417-3\n \n \nGeneral  Instructions   ●  All  rough  work  should  be  done  in  the  answer  script.  Do  not  write  or  scribble  on  the  question  paper  \nexcept\n \nyour\n \nregistration\n \nnumber.\n \n ●  Verify  the  Course  code  /  Course  title  &  number  of  pages  of  questions  in  the  question  paper.  \n●\n \nEnsure\n \nyour\n \nmobile\n \nphone\n \

In [4]:
##textsplitting into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split{len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"example chunk")
        print(f"content:{split_docs[0].page_content[:200]}...")
        print(f"metdata:{split_docs[0].metadata}")
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)

Split29 documents into 38 chunks
example chunk
content:CHRIST  (Deemed  to  be  University),  Bangalore  –  560  029   Department  of  Computer  Science   END-TRIMESTER  EXAMINATION  MARCH-  2026   PG  III  Trimester   
Programme  Name:  MSAIM  Max.  Mark...
metdata:{'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Indrasena_2548561_ETE-DL QP', 'source': '..\\data\\pdfs\\Indrasena_2548561_ETE-DL QP.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Indrasena_2548561_ETE-DL QP.pdf', 'file_type': 'pdf'}


## embedding and vectorstore db

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Any,Tuple,Dict
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
## handles the embedding generation using setence transformer
class EmbeddingManager:
    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        """
        intilaize the embedding manager
        Args:
            Model name: hugging face model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()## its going to load the model
     ## loading the sentence transformer   
    def _load_model(self):# why we are using _ in the begginig.. it is a protected functionand it will be accessible only inside this class
        try:
            print(f"loading embedding model{self.model_name}")
            self.model =SentenceTransformer(self.model_name)
            print("model loaded successfully. embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"error loading model {self.model_name}:{e}")
            raise

    ## generating the embedding for the list of texts
    def generate_embeddings(self,texts:List[str])->np.ndarray:
        """
        ARGS:
            texts:list of text strings to emb
        return:
            numpy array od embeddings wiht shape (len(texts),embedding_dim)
        
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"generating embeddings for {len(texts)}texts")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"generated embeddings wiht shape :{embeddings.shape}")
        return embeddings

##def get_embedding_dimension(self) ->int:
# ##   """get the embedding dimesnion of the model"""
# ## if not self.model():
# ## return self.model.get_sentence_embedding_dimension()
# ##     raise valueError("model not loaded")


## intilaize the embedding manager
embedding_manager=EmbeddingManager()

loading embedding modelall-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1429.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded successfully. embedding dimension: {self.model.get_sentence_embedding_dimension()}


## vector store

In [8]:
## manages the document embeddings in a chromadb vector store
## persist_direcotory is whatever vector store is creating it is saving in harddisk 
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        """
        initilaze the vector store 
        Args:
            collection_name : name of the chromadb collection
            persist_directory: directory to persist the vectorspace 
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
# intializing the chroma db client and collection
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

        #get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"pdf documnet embedding for rag"}
            )
            
            print(f"Vector store initialized . collection :{self.collection_name}")
            print(f"existing documents in collection :{self.collection.count()}")
        except Exception as e:
            print(f"error inittiliazing in vector store:{e} ")
            raise

    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
            """add documents and embeddings to vectore store 
            args:
                documents:list of lanchain documents
                embeddings:correspoinding embeddings for the documents
            """
            if len(documents)!= len(embeddings):
                raise ValueError("no of documents must match with no of embeddings")
            print(f"adding{len(documents)}documents to vector store..")
            #prepare db for chromadb
            ids=[]
            metadatas=[]
            documents_text=[]
            embeddings_list=[]
            for i ,(doc,embedding) in enumerate(zip(documents,embeddings)):
                doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
                ids.append(doc_id)
                #prepare metdata
                metadata=dict(doc.metadata)
                metadata['doc_index']=i
                metadata['content_length']=len(doc.page_content)
                metadatas.append(metadata)
            
                documents_text.append(doc.page_content)
                embeddings_list.append(embedding.tolist())
           # add to collection
            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )
                print(f"succeddfully added  {len(documents)}documents to vector store")
                print(f"total documents in collection :{self.collection.count()}")

            except Exception as e:
                print(f"error add ing to documents to vectore store :{e}")
                raise
vectorstore=VectorStore()
vectorstore
        

Vector store initialized . collection :pdf_documents
existing documents in collection :38


In [9]:
chunks

[Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Indrasena_2548561_ETE-DL QP', 'source': '..\\data\\pdfs\\Indrasena_2548561_ETE-DL QP.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Indrasena_2548561_ETE-DL QP.pdf', 'file_type': 'pdf'}, page_content='CHRIST  (Deemed  to  be  University),  Bangalore  –  560  029   Department  of  Computer  Science   END-TRIMESTER  EXAMINATION  MARCH-  2026   PG  III  Trimester   \nProgramme  Name:  MSAIM  Max.  Marks:  100  Course  Name:  -  Deep  Learning  Time:  3  Hrs  \nCourse\n \nCode:\n \nMAI417-3\n \n \nGeneral  Instructions   ●  All  rough  work  should  be  done  in  the  answer  script.  Do  not  write  or  scribble  on  the  question  paper  \nexcept\n \nyour\n \nregistration\n \nnumber.\n \n ●  Verify  the  Course  code  /  Course  title  &  number  of  pages  of  questions  in  the  question  paper.  \n●\n \nEnsure\n \nyour\n \nmobile\n \nphone\n \

In [10]:
# we will extract the text from the chunk and will do embedding
texts=[doc.page_content for doc in chunks] 
# generatet the embediings
embeddings =embedding_manager.generate_embeddings(texts)
#store in the vectore db 
vectorstore.add_documents(chunks,embeddings)

generating embeddings for 38texts


Batches: 100%|██████████| 2/2 [00:05<00:00,  2.92s/it]


generated embeddings wiht shape :(38, 384)
adding38documents to vector store..
succeddfully added  38documents to vector store
total documents in collection :76


# retriever pipeline from vector store

In [14]:
class RagRetriever:
    def __init__(self,vector_store:VectorStore,embedding_manger:EmbeddingManager):
        """
        initialize the retriever 
        args:
            vector_store:vector store containing doucment embeddings
            embedding_manager:manger for generating for queryembeddings
        """
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self,query:str,top_k:int =5,score_threshold : float =0.0)->List[Dict[str,Any]]:
        """
        Retrieve relevant document for a query
        Args:
            query:the search query
            top_k=no of top results to return
            score_threshold=minimum similarity score threshold

        returns:
            list of dictionries containing retrived documnets and metadata
        """
        print(f"retrieving document for query:'{query}'")
        print(f"top_k :{top_k} , score_thresold :{score_threshold}")
    # generate query embedding
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            #process results
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i ,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    #convert distance to similarity score (chromadb uses cosine distance)
                    similarity_score=1-distance
                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("no documents found")
            return retrieved_docs
        except Exception as e:
            print(f"error during retrieval {e}")
            return[]
ragretriever=RagRetriever(vectorstore,embedding_manager)
ragretriever

In [15]:
ragretriever.retrieve("waht is hackathon challenge ")

retrieving document for query:'waht is hackathon challenge '
top_k :5 , score_thresold :0.0
generating embeddings for 1texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 45.46it/s]


generated embeddings wiht shape :(1, 384)
retrieved 2 documents (after filtering)


[{'id': 'doc_9d86bac4_2',
  'content': 'problems.\n \nThis\n \nhackathon\n \nbased\n \nexamination\n \nchallenges\n \nyou\n \nto\n \ndesign\n \nand\n \nimplement\n \nan\n \nintelligent\n \nsystem\n \nusing\n \ntechniques\n \nlearned\n \nthroughout\n \nthe\n \ncourse.\n \n \nYour  task  is  to  build  an  end  to  end  deep  learning  solution  using  the  dataset  provided  during  the  \nexamination.\n \nThe\n \nsolution\n \nshould\n \ndemonstrate\n \nyour\n \nability\n \nto\n \nunderstand\n \na\n \nproblem,\n \nmodel\n \nit\n \nmathematically,\n \ndesign\n \nan\n \nappropriate\n \ndeep\n \nlearning\n \narchitecture,\n \nand\n \ndevelop\n \na\n \nworking\n \nprototype.\n \n \nYou  may  use  frameworks  such  as  TensorFlow,  Keras,  or  PyTorch .   \nThis  challenge  evaluates  your  technical  understanding,  creativity,  and  ability  to  translate  deep  \nlearning\n \nconcepts\n \ninto\n \npractical\n \napplications\n.\n \nHackathon  Challenge   \nDesign  and  implement  an  end  

### integration vector db context pipeline with llm output

In [ ]:
### simple rag pipeline wiht groq llm
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv
load_dotenv()

### intialize the groq llm 
groq_api_key=os.getenv("GROQ_API_KEY")
llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024) 

## simple rag function : retrieve context +generate response